In [1]:
import sys
from pathlib import Path
sys.path.append(str(Path().resolve().parents[1]))

In [2]:
from Libraries.inference_training import Configuration, ImageDataset
from Libraries.inference_training import initCudaEnvironment, createTransforms
from Libraries.inference_training import drawImageAndFeatureMasks
from Libraries.inference_training import exportOnnxModel, writeONNXMeta, loadONNX
from Libraries.inference_training import trainModel, saveModel, loadModel
from Libraries.inference_training import createModelInstance, testInference
from Libraries.inference_training import testInferenceWithIoU
from paths import MASKRCNN_RESEARCH_TRAIN, MASKRCNN_RESEARCH_TEST

In [3]:
config = Configuration()
print("Device: " + str(config.device))

config = Configuration()
config.setDatasetPaths(trainPath=MASKRCNN_RESEARCH_TRAIN, testPath=MASKRCNN_RESEARCH_TEST)  
config.setModelName("mytrainedmodel")
config.setInputSizes(250, 250)
config.setModelInfo(channels=3, numClasses=3, bboxOverlap=True, bboxPerImage=250, reuseModel=True)
config.setOnnxMetaData(scoreThreshold=0.2, maskThreshold=0.3, strideFraction=0.5)

model = createModelInstance(config)
loadModel(config, model, path=config.getPytorchModelFileName())
model.to(config.device)
model.eval()

testDataset = ImageDataset(config, False, createTransforms(False))

Device: cpu
Detections per image 250
in features mask: 256


In [4]:
ious_all = []
for i in range(len(testDataset)):
    ious = testInferenceWithIoU(config, testDataset, model, i)
    ious_all.extend(ious)

map_50 = sum(iou >= 0.5 for iou in ious_all) / len(ious_all)
print(f"mAP@0.5: {map_50:.3f}")


mAP@0.5: 0.119
